# Logistic Regression

In [1]:
import pandas as pd
import numpy as np
import pickle
import time
import json
 
from scipy import sparse
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve,
    average_precision_score,
    log_loss,
    matthews_corrcoef,
    cohen_kappa_score,
)
 
CSV_PATH = "tfidf_vectors.csv"
LABEL_COL = "label"
MODEL_OUT = "logistic_regression_model.pkl"
METRICS_OUT = "evaluation_metrics.pkl"
RANDOM_STATE = 42


### 1. Load data efficiently

In [2]:
print("Loading CSV (float32 to save memory)...")
t0 = time.time()
 
# Peek at columns first so we can build a dtype map without reading twice
header = pd.read_csv(CSV_PATH, nrows=0)
feature_cols = [c for c in header.columns if c != LABEL_COL]
 
dtype_map = {c: np.float32 for c in feature_cols}
dtype_map[LABEL_COL] = np.int8  # assumes binary/int labels; adjust if needed
 
df = pd.read_csv(CSV_PATH, dtype=dtype_map)
print(f"Loaded {df.shape[0]:,} rows x {df.shape[1]:,} cols in {time.time()-t0:.1f}s")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")
 
X_dense = df[feature_cols]
y = df[LABEL_COL].values

# NOTE: df is NOT deleted here (unlike before) — we need it a little longer
# for the leakage check below. It gets freed right after that.


Loading CSV (float32 to save memory)...
Loaded 44,682 rows x 30,009 cols in 813.8s
Memory usage: 5.36 GB


### 1b. Data Leakage Check

Every model previously scored 0.99-1.00 across the board — that never happens by chance across five unrelated algorithm families. The likely cause: the `subject_tfidf__*` columns encode which news source/section an article was scraped from, which correlates almost perfectly with the `label` (real articles were pulled from Reuters `politicsNews`/`worldnews`, fake ones from other feeds with different subject tags). That makes `subject` a direct proxy for the label instead of a genuine text signal. We check that below before deciding what to drop.

In [3]:
print("=" * 60)
print("DATA LEAKAGE CHECK")
print("=" * 60)

subject_cols = [c for c in feature_cols if c.startswith("subject_tfidf__")]
print(f"Found {len(subject_cols)} subject_tfidf__ columns: {subject_cols}\n")

# Correlation of each subject column with the label
corr_check = (
    df[subject_cols + [LABEL_COL]]
    .corr()[LABEL_COL]
    .drop(LABEL_COL)
    .sort_values(key=abs, ascending=False)
)
print("Correlation of subject_tfidf__ columns with label:")
print(corr_check)

# Crosstab: does a nonzero subject value perfectly predict the label?
print("\nCrosstab (subject column present vs label):")
for col in subject_cols:
    ct = pd.crosstab(df[col] > 0, df[LABEL_COL])
    print(f"\n{col}:")
    print(ct)

# Duplicate rows across all features can also inflate scores via
# train/test leakage (near-identical rows split across both sets)
n_dupes = df.duplicated(subset=feature_cols).sum()
print(f"\nDuplicate feature rows in full dataset: {n_dupes:,} ({n_dupes/len(df):.2%})")


DATA LEAKAGE CHECK
Found 8 subject_tfidf__ columns: ['subject_tfidf__east', 'subject_tfidf__government', 'subject_tfidf__left', 'subject_tfidf__middle', 'subject_tfidf__news', 'subject_tfidf__politics', 'subject_tfidf__politicsnews', 'subject_tfidf__worldnews']

Correlation of subject_tfidf__ columns with label:
subject_tfidf__news           -0.635813
subject_tfidf__politicsnews    0.609088
subject_tfidf__worldnews       0.564624
subject_tfidf__politics       -0.403985
subject_tfidf__left           -0.316381
subject_tfidf__government     -0.181379
subject_tfidf__middle         -0.126524
subject_tfidf__east           -0.126524
Name: label, dtype: float64

Crosstab (subject column present vs label):

subject_tfidf__east:
label                    0      1
subject_tfidf__east              
False                22697  21207
True                   778      0

subject_tfidf__government:
label                          0      1
subject_tfidf__government              
False                      

### 1c. Fix: drop the leaking `subject_tfidf__*` columns

If the correlation/crosstab above confirms `subject_tfidf__*` near-perfectly predicts `label`, these columns are metadata about data collection, not a genuine text feature — leaving them in means the model learns to read the scrape source instead of detecting fake news. We drop them so training relies only on `title_tfidf__*` and `content_tfidf__*` (the actual article text).

In [4]:
print("Dropping subject_tfidf__ columns to remove label leakage...")
leak_cols = [c for c in feature_cols if c.startswith("subject_tfidf__")]
feature_cols = [c for c in feature_cols if c not in leak_cols]

# Reselect X_dense to only the remaining (non-leaking) columns
X_dense = X_dense[feature_cols]

print(f"Dropped {len(leak_cols)} columns: {leak_cols}")
print(f"Remaining features: {len(feature_cols):,}")

# Now safe to free df
del df


Dropping subject_tfidf__ columns to remove label leakage...
Dropped 8 columns: ['subject_tfidf__east', 'subject_tfidf__government', 'subject_tfidf__left', 'subject_tfidf__middle', 'subject_tfidf__news', 'subject_tfidf__politics', 'subject_tfidf__politicsnews', 'subject_tfidf__worldnews']
Remaining features: 30,000


### 2. Convert to sparse — TF-IDF is mostly zeros, this is the big memory win

In [5]:
print("Converting to sparse matrix...")
X = sparse.csr_matrix(X_dense.values)
del X_dense
print(f"Sparse matrix: {X.shape}, density = {X.nnz / (X.shape[0]*X.shape[1]):.4%}")
 
feature_names = feature_cols  # keep for later reference


Converting to sparse matrix...
Sparse matrix: (44682, 30000), density = 0.4933%


### 3. Train/test split

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")


Train: (35745, 30000), Test: (8937, 30000)


### 4. Train Logistic Regression

In [7]:
print("Training Logistic Regression...")
t0 = time.time()
 
model = LogisticRegression(
    max_iter=1000,
    solver="saga",       # handles large sparse data well, supports L1/L2
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
model.fit(X_train, y_train)
print(f"Training took {time.time()-t0:.1f}s")


Training Logistic Regression...


C:\Users\DELL\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Training took 2.9s


### 5. Predictions

In [8]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]  # binary classification assumed


### 6. Evaluation metrics

In [9]:
print("\n" + "=" * 60)
print("EVALUATION METRICS")
print("=" * 60)
 
metrics = {}
metrics["accuracy"] = accuracy_score(y_test, y_pred)
metrics["precision"] = precision_score(y_test, y_pred, average="weighted", zero_division=0)
metrics["recall"] = recall_score(y_test, y_pred, average="weighted", zero_division=0)
metrics["f1_score"] = f1_score(y_test, y_pred, average="weighted", zero_division=0)
metrics["roc_auc"] = roc_auc_score(y_test, y_proba)
metrics["average_precision"] = average_precision_score(y_test, y_proba)
metrics["log_loss"] = log_loss(y_test, y_proba)
metrics["matthews_corrcoef"] = matthews_corrcoef(y_test, y_pred)
metrics["cohen_kappa"] = cohen_kappa_score(y_test, y_pred)
 
cm = confusion_matrix(y_test, y_pred)
metrics["confusion_matrix"] = cm.tolist()
 
report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
metrics["classification_report"] = report
 
fpr, tpr, _ = roc_curve(y_test, y_proba)
prec_curve, rec_curve, _ = precision_recall_curve(y_test, y_proba)
metrics["roc_curve"] = {"fpr": fpr.tolist(), "tpr": tpr.tolist()}
metrics["pr_curve"] = {"precision": prec_curve.tolist(), "recall": rec_curve.tolist()}
 
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"{k:>22}: {v:.4f}")
 
print("\nConfusion Matrix:")
print(cm)
 
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))



EVALUATION METRICS
              accuracy: 0.9927
             precision: 0.9927
                recall: 0.9927
              f1_score: 0.9927
               roc_auc: 0.9995
     average_precision: 0.9995
              log_loss: 0.0621
     matthews_corrcoef: 0.9854
           cohen_kappa: 0.9854

Confusion Matrix:
[[4666   29]
 [  36 4206]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      4695
           1       0.99      0.99      0.99      4242

    accuracy                           0.99      8937
   macro avg       0.99      0.99      0.99      8937
weighted avg       0.99      0.99      0.99      8937



### 7. Save model + metrics + feature names

In [10]:
print(f"\nSaving model to {MODEL_OUT} ...")
with open(MODEL_OUT, "wb") as f:
    pickle.dump(
        {
            "model": model,
            "feature_names": feature_names,
        },
        f,
    )
 
print(f"Saving metrics to {METRICS_OUT} ...")
with open(METRICS_OUT, "wb") as f:
    pickle.dump(metrics, f)
 
print("Done.")



Saving model to logistic_regression_model.pkl ...
Saving metrics to evaluation_metrics.pkl ...
Done.


# Naive Bayes

In [11]:
import pickle
import time
 
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve,
    average_precision_score,
    log_loss,
    matthews_corrcoef,
    cohen_kappa_score,
)
 
NB_MODEL_OUT = "naive_bayes_model.pkl"
NB_METRICS_OUT = "nb_evaluation_metrics.pkl"


### 1. Train Multinomial Naive Bayes

In [12]:
print("Training Multinomial Naive Bayes...")
t0 = time.time()
 
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)
print(f"Training took {time.time()-t0:.1f}s")


Training Multinomial Naive Bayes...
Training took 0.1s


### 2. Predictions

In [13]:
y_pred_nb = nb_model.predict(X_test)
y_proba_nb = nb_model.predict_proba(X_test)[:, 1]


### 3. Evaluation metrics

In [14]:
print("\n" + "=" * 60)
print("NAIVE BAYES - EVALUATION METRICS")
print("=" * 60)
 
nb_metrics = {}
nb_metrics["accuracy"] = accuracy_score(y_test, y_pred_nb)
nb_metrics["precision"] = precision_score(y_test, y_pred_nb, average="weighted", zero_division=0)
nb_metrics["recall"] = recall_score(y_test, y_pred_nb, average="weighted", zero_division=0)
nb_metrics["f1_score"] = f1_score(y_test, y_pred_nb, average="weighted", zero_division=0)
nb_metrics["roc_auc"] = roc_auc_score(y_test, y_proba_nb)
nb_metrics["average_precision"] = average_precision_score(y_test, y_proba_nb)
nb_metrics["log_loss"] = log_loss(y_test, y_proba_nb)
nb_metrics["matthews_corrcoef"] = matthews_corrcoef(y_test, y_pred_nb)
nb_metrics["cohen_kappa"] = cohen_kappa_score(y_test, y_pred_nb)
 
cm_nb = confusion_matrix(y_test, y_pred_nb)
nb_metrics["confusion_matrix"] = cm_nb.tolist()
 
report_nb = classification_report(y_test, y_pred_nb, output_dict=True, zero_division=0)
nb_metrics["classification_report"] = report_nb
 
fpr_nb, tpr_nb, _ = roc_curve(y_test, y_proba_nb)
prec_curve_nb, rec_curve_nb, _ = precision_recall_curve(y_test, y_proba_nb)
nb_metrics["roc_curve"] = {"fpr": fpr_nb.tolist(), "tpr": tpr_nb.tolist()}
nb_metrics["pr_curve"] = {"precision": prec_curve_nb.tolist(), "recall": rec_curve_nb.tolist()}
 
for k, v in nb_metrics.items():
    if isinstance(v, float):
        print(f"{k:>22}: {v:.4f}")
 
print("\nConfusion Matrix:")
print(cm_nb)
 
print("\nClassification Report:")
print(classification_report(y_test, y_pred_nb, zero_division=0))



NAIVE BAYES - EVALUATION METRICS
              accuracy: 0.9465
             precision: 0.9465
                recall: 0.9465
              f1_score: 0.9465
               roc_auc: 0.9877
     average_precision: 0.9855
              log_loss: 0.1475
     matthews_corrcoef: 0.8927
           cohen_kappa: 0.8927

Confusion Matrix:
[[4478  217]
 [ 261 3981]]

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.95      0.95      4695
           1       0.95      0.94      0.94      4242

    accuracy                           0.95      8937
   macro avg       0.95      0.95      0.95      8937
weighted avg       0.95      0.95      0.95      8937



### 4. Save model + metrics

In [15]:
print(f"\nSaving model to {NB_MODEL_OUT} ...")
with open(NB_MODEL_OUT, "wb") as f:
    pickle.dump(
        {
            "model": nb_model,
            "feature_names": feature_names,
        },
        f,
    )
 
print(f"Saving metrics to {NB_METRICS_OUT} ...")
with open(NB_METRICS_OUT, "wb") as f:
    pickle.dump(nb_metrics, f)
 
print("Done.")



Saving model to naive_bayes_model.pkl ...
Saving metrics to nb_evaluation_metrics.pkl ...
Done.


### 5. Quick side-by-side comparison with Logistic Regression

In [16]:
try:
    print("\n" + "=" * 60)
    print("LOGISTIC REGRESSION vs NAIVE BAYES")
    print("=" * 60)
    for k in ["accuracy", "precision", "recall", "f1_score", "roc_auc"]:
        print(f"{k:>12} | LR: {metrics[k]:.4f} | NB: {nb_metrics[k]:.4f}")
except NameError:
    pass


LOGISTIC REGRESSION vs NAIVE BAYES
    accuracy | LR: 0.9927 | NB: 0.9465
   precision | LR: 0.9927 | NB: 0.9465
      recall | LR: 0.9927 | NB: 0.9465
    f1_score | LR: 0.9927 | NB: 0.9465
     roc_auc | LR: 0.9995 | NB: 0.9877


# SVM

In [17]:
"""
Linear SVM on the same TF-IDF sparse data.

Run this AFTER the logistic regression script in the same notebook session —
it reuses X_train, X_test, y_train, y_test, feature_names already in memory.
No need to reload the CSV.

Uses LinearSVC, not kernel SVC:
  - SVC (RBF/poly/etc.) scales ~O(n^2) to O(n^3) with sample count and is
    not built for high-dimensional sparse TF-IDF data at this size — it can
    take hours or run out of memory.
  - LinearSVC uses liblinear internally, built for exactly this case:
    large, sparse, high-dimensional text features.

LinearSVC has no predict_proba by default (it's a max-margin classifier,
not probabilistic), so it's wrapped in CalibratedClassifierCV to get
probability estimates for ROC-AUC / log-loss / average-precision.
"""

import pickle
import time

from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve,
    average_precision_score,
    log_loss,
    matthews_corrcoef,
    cohen_kappa_score,
)

SVM_MODEL_OUT = "svm_model.pkl"
SVM_METRICS_OUT = "svm_evaluation_metrics.pkl"

# ---------------------------------------------------------------------------
# 1. Train Linear SVM, calibrated for probability estimates
# ---------------------------------------------------------------------------
print("Training LinearSVC...")
t0 = time.time()

base_svm = LinearSVC(
    max_iter=5000,   # liblinear can need more iterations to converge on TF-IDF
    dual="auto",     # let sklearn pick based on n_samples vs n_features
    random_state=42,
)

# CalibratedClassifierCV wraps LinearSVC to produce predict_proba via
# 5-fold internal cross-validation (Platt-style calibration). This costs
# extra training time (~5x) since it trains 5 internal SVMs — worth it
# for the full metric suite, but drop the wrapper and use decision_function
# directly if you only need accuracy/precision/recall/F1/confusion matrix.
svm_model = CalibratedClassifierCV(base_svm, cv=5)
svm_model.fit(X_train, y_train)
print(f"Training took {time.time()-t0:.1f}s")

# ---------------------------------------------------------------------------
# 2. Predictions
# ---------------------------------------------------------------------------
y_pred_svm = svm_model.predict(X_test)
y_proba_svm = svm_model.predict_proba(X_test)[:, 1]

# ---------------------------------------------------------------------------
# 3. Evaluation metrics (same suite as before, for comparison)
# ---------------------------------------------------------------------------
print("\n" + "=" * 60)
print("SVM - EVALUATION METRICS")
print("=" * 60)

svm_metrics = {}
svm_metrics["accuracy"] = accuracy_score(y_test, y_pred_svm)
svm_metrics["precision"] = precision_score(y_test, y_pred_svm, average="weighted", zero_division=0)
svm_metrics["recall"] = recall_score(y_test, y_pred_svm, average="weighted", zero_division=0)
svm_metrics["f1_score"] = f1_score(y_test, y_pred_svm, average="weighted", zero_division=0)
svm_metrics["roc_auc"] = roc_auc_score(y_test, y_proba_svm)
svm_metrics["average_precision"] = average_precision_score(y_test, y_proba_svm)
svm_metrics["log_loss"] = log_loss(y_test, y_proba_svm)
svm_metrics["matthews_corrcoef"] = matthews_corrcoef(y_test, y_pred_svm)
svm_metrics["cohen_kappa"] = cohen_kappa_score(y_test, y_pred_svm)

cm_svm = confusion_matrix(y_test, y_pred_svm)
svm_metrics["confusion_matrix"] = cm_svm.tolist()

report_svm = classification_report(y_test, y_pred_svm, output_dict=True, zero_division=0)
svm_metrics["classification_report"] = report_svm

fpr_svm, tpr_svm, _ = roc_curve(y_test, y_proba_svm)
prec_curve_svm, rec_curve_svm, _ = precision_recall_curve(y_test, y_proba_svm)
svm_metrics["roc_curve"] = {"fpr": fpr_svm.tolist(), "tpr": tpr_svm.tolist()}
svm_metrics["pr_curve"] = {"precision": prec_curve_svm.tolist(), "recall": rec_curve_svm.tolist()}

for k, v in svm_metrics.items():
    if isinstance(v, float):
        print(f"{k:>22}: {v:.4f}")

print("\nConfusion Matrix:")
print(cm_svm)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_svm, zero_division=0))

# ---------------------------------------------------------------------------
# 4. Save model + metrics
# ---------------------------------------------------------------------------
print(f"\nSaving model to {SVM_MODEL_OUT} ...")
with open(SVM_MODEL_OUT, "wb") as f:
    pickle.dump(
        {
            "model": svm_model,
            "feature_names": feature_names,
        },
        f,
    )

print(f"Saving metrics to {SVM_METRICS_OUT} ...")
with open(SVM_METRICS_OUT, "wb") as f:
    pickle.dump(svm_metrics, f)

print("Done.")

# ---------------------------------------------------------------------------
# 5. Quick side-by-side comparison, if earlier metrics dicts are in memory
# ---------------------------------------------------------------------------
try:
    print("\n" + "=" * 60)
    print("LOGISTIC REGRESSION vs NAIVE BAYES vs SVM")
    print("=" * 60)
    for k in ["accuracy", "precision", "recall", "f1_score", "roc_auc"]:
        row = f"{k:>12} | LR: {metrics[k]:.4f}"
        try:
            row += f" | NB: {nb_metrics[k]:.4f}"
        except NameError:
            pass
        row += f" | SVM: {svm_metrics[k]:.4f}"
        print(row)
except NameError:
    pass  # earlier metrics dicts aren't in memory — skip comparison

Training LinearSVC...
Training took 3.9s

SVM - EVALUATION METRICS
              accuracy: 0.9960
             precision: 0.9960
                recall: 0.9960
              f1_score: 0.9960
               roc_auc: 0.9999
     average_precision: 0.9998
              log_loss: 0.0143
     matthews_corrcoef: 0.9919
           cohen_kappa: 0.9919

Confusion Matrix:
[[4679   16]
 [  20 4222]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      4695
           1       1.00      1.00      1.00      4242

    accuracy                           1.00      8937
   macro avg       1.00      1.00      1.00      8937
weighted avg       1.00      1.00      1.00      8937


Saving model to svm_model.pkl ...
Saving metrics to svm_evaluation_metrics.pkl ...
Done.

LOGISTIC REGRESSION vs NAIVE BAYES vs SVM
    accuracy | LR: 0.9927 | NB: 0.9465 | SVM: 0.9960
   precision | LR: 0.9927 | NB: 0.9465 | SVM: 0.9960
      recall | LR:

# Random Forest

In [18]:
"""
Random Forest on the same TF-IDF sparse data.

Run this AFTER the logistic regression script in the same notebook session —
it reuses X_train, X_test, y_train, y_test, feature_names already in memory.
No need to reload the CSV.

Important difference from LR/NB/SVM: Random Forest has no true sparse-native
training path. sklearn's RandomForestClassifier accepts a sparse matrix as
input, but each tree still evaluates dense-like splits internally, so with
thousands of TF-IDF columns this will be noticeably slower and more memory-
hungry than the previous models. Mitigations applied below:
  - max_depth capped (unbounded trees on high-dimensional sparse data can
    each grow huge and eat RAM)
  - n_estimators kept moderate (can raise later once you see timing/memory)
  - n_jobs=-1 to parallelize across cores
  - max_features="sqrt" to reduce per-split cost on thousands of columns
"""

import pickle
import time
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve,
    average_precision_score,
    log_loss,
    matthews_corrcoef,
    cohen_kappa_score,
)

RF_MODEL_OUT = "random_forest_model.pkl"
RF_METRICS_OUT = "rf_evaluation_metrics.pkl"

# ---------------------------------------------------------------------------
# 1. Train Random Forest
# ---------------------------------------------------------------------------
print("Training Random Forest...")
t0 = time.time()

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=30,          # cap depth — unbounded trees on thousands of
                            # sparse TF-IDF columns can blow up memory/time
    max_features="sqrt",   # standard for high-dimensional data, cheaper splits
    n_jobs=-1,
    random_state=42,
    class_weight="balanced",  # helpful if label classes are imbalanced
)
rf_model.fit(X_train, y_train)
print(f"Training took {time.time()-t0:.1f}s")

# ---------------------------------------------------------------------------
# 2. Predictions
# ---------------------------------------------------------------------------
y_pred_rf = rf_model.predict(X_test)
y_proba_rf = rf_model.predict_proba(X_test)[:, 1]

# ---------------------------------------------------------------------------
# 3. Evaluation metrics (same suite as before, for comparison)
# ---------------------------------------------------------------------------
print("\n" + "=" * 60)
print("RANDOM FOREST - EVALUATION METRICS")
print("=" * 60)

rf_metrics = {}
rf_metrics["accuracy"] = accuracy_score(y_test, y_pred_rf)
rf_metrics["precision"] = precision_score(y_test, y_pred_rf, average="weighted", zero_division=0)
rf_metrics["recall"] = recall_score(y_test, y_pred_rf, average="weighted", zero_division=0)
rf_metrics["f1_score"] = f1_score(y_test, y_pred_rf, average="weighted", zero_division=0)
rf_metrics["roc_auc"] = roc_auc_score(y_test, y_proba_rf)
rf_metrics["average_precision"] = average_precision_score(y_test, y_proba_rf)
rf_metrics["log_loss"] = log_loss(y_test, y_proba_rf)
rf_metrics["matthews_corrcoef"] = matthews_corrcoef(y_test, y_pred_rf)
rf_metrics["cohen_kappa"] = cohen_kappa_score(y_test, y_pred_rf)

cm_rf = confusion_matrix(y_test, y_pred_rf)
rf_metrics["confusion_matrix"] = cm_rf.tolist()

report_rf = classification_report(y_test, y_pred_rf, output_dict=True, zero_division=0)
rf_metrics["classification_report"] = report_rf

fpr_rf, tpr_rf, _ = roc_curve(y_test, y_proba_rf)
prec_curve_rf, rec_curve_rf, _ = precision_recall_curve(y_test, y_proba_rf)
rf_metrics["roc_curve"] = {"fpr": fpr_rf.tolist(), "tpr": tpr_rf.tolist()}
rf_metrics["pr_curve"] = {"precision": prec_curve_rf.tolist(), "recall": rec_curve_rf.tolist()}

for k, v in rf_metrics.items():
    if isinstance(v, float):
        print(f"{k:>22}: {v:.4f}")

print("\nConfusion Matrix:")
print(cm_rf)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf, zero_division=0))

# ---------------------------------------------------------------------------
# 4. Feature importances — a bonus RF gives you that LR/NB/SVM don't
#    directly hand you in this form
# ---------------------------------------------------------------------------
importances = rf_model.feature_importances_
top_idx = np.argsort(importances)[::-1][:20]
print("\nTop 20 most important features:")
for i in top_idx:
    print(f"  {feature_names[i]:<40} {importances[i]:.4f}")

rf_metrics["feature_importances"] = dict(zip(feature_names, importances.tolist()))

# ---------------------------------------------------------------------------
# 5. Save model + metrics
# ---------------------------------------------------------------------------
print(f"\nSaving model to {RF_MODEL_OUT} ...")
with open(RF_MODEL_OUT, "wb") as f:
    pickle.dump(
        {
            "model": rf_model,
            "feature_names": feature_names,
        },
        f,
    )

print(f"Saving metrics to {RF_METRICS_OUT} ...")
with open(RF_METRICS_OUT, "wb") as f:
    pickle.dump(rf_metrics, f)

print("Done.")

# ---------------------------------------------------------------------------
# 6. Comparison across all models trained so far, if their metrics dicts
#    are still in memory
# ---------------------------------------------------------------------------
try:
    print("\n" + "=" * 60)
    print("LOGISTIC REGRESSION vs NAIVE BAYES vs SVM vs RANDOM FOREST")
    print("=" * 60)
    for k in ["accuracy", "precision", "recall", "f1_score", "roc_auc"]:
        row = f"{k:>12} | LR: {metrics[k]:.4f}"
        try:
            row += f" | NB: {nb_metrics[k]:.4f}"
        except NameError:
            pass
        try:
            row += f" | SVM: {svm_metrics[k]:.4f}"
        except NameError:
            pass
        row += f" | RF: {rf_metrics[k]:.4f}"
        print(row)
except NameError:
    pass  # earlier metrics dicts aren't in memory — skip comparison

Training Random Forest...
Training took 28.7s

RANDOM FOREST - EVALUATION METRICS
              accuracy: 0.9932
             precision: 0.9932
                recall: 0.9932
              f1_score: 0.9932
               roc_auc: 0.9997
     average_precision: 0.9997
              log_loss: 0.2264
     matthews_corrcoef: 0.9863
           cohen_kappa: 0.9863

Confusion Matrix:
[[4663   32]
 [  29 4213]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      4695
           1       0.99      0.99      0.99      4242

    accuracy                           0.99      8937
   macro avg       0.99      0.99      0.99      8937
weighted avg       0.99      0.99      0.99      8937


Top 20 most important features:
  content_tfidf__reuters                   0.0979
  content_tfidf__said                      0.0363
  title_tfidf__video                       0.0267
  content_tfidf__image                     0.0189
  content

# XGBoost

In [19]:
"""
XGBoost on the same TF-IDF sparse data.

Run this AFTER the logistic regression script in the same notebook session —
it reuses X_train, X_test, y_train, y_test, feature_names already in memory.
No need to reload the CSV.

Unlike Random Forest, XGBoost's tree-building algorithm has genuine native
support for sparse matrices (it exploits sparsity directly when finding
splits), so it typically trains faster than RF on TF-IDF data despite often
matching or beating it on accuracy. Uses hist tree_method, which is the
efficient histogram-based approach — the default in recent xgboost versions
but set explicitly here since it matters most for large sparse data.
"""

import pickle
import time
import numpy as np

import xgboost as xgb
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve,
    average_precision_score,
    log_loss,
    matthews_corrcoef,
    cohen_kappa_score,
)

XGB_MODEL_OUT = "xgboost_model.pkl"
XGB_METRICS_OUT = "xgb_evaluation_metrics.pkl"

# ---------------------------------------------------------------------------
# 1. Train XGBoost
# ---------------------------------------------------------------------------
print("Training XGBoost...")
t0 = time.time()

# Handle class imbalance the XGBoost way, if applicable
n_pos = int((y_train == 1).sum())
n_neg = int((y_train == 0).sum())
scale_pos_weight = n_neg / n_pos if n_pos > 0 else 1.0

xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    tree_method="hist",        # efficient histogram-based splits, key for
                                # large sparse high-dimensional data
    n_jobs=-1,
    random_state=42,
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight,
)
xgb_model.fit(X_train, y_train)
print(f"Training took {time.time()-t0:.1f}s")

# ---------------------------------------------------------------------------
# 2. Predictions
# ---------------------------------------------------------------------------
y_pred_xgb = xgb_model.predict(X_test)
y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

# ---------------------------------------------------------------------------
# 3. Evaluation metrics (same suite as before, for comparison)
# ---------------------------------------------------------------------------
print("\n" + "=" * 60)
print("XGBOOST - EVALUATION METRICS")
print("=" * 60)

xgb_metrics = {}
xgb_metrics["accuracy"] = accuracy_score(y_test, y_pred_xgb)
xgb_metrics["precision"] = precision_score(y_test, y_pred_xgb, average="weighted", zero_division=0)
xgb_metrics["recall"] = recall_score(y_test, y_pred_xgb, average="weighted", zero_division=0)
xgb_metrics["f1_score"] = f1_score(y_test, y_pred_xgb, average="weighted", zero_division=0)
xgb_metrics["roc_auc"] = roc_auc_score(y_test, y_proba_xgb)
xgb_metrics["average_precision"] = average_precision_score(y_test, y_proba_xgb)
xgb_metrics["log_loss"] = log_loss(y_test, y_proba_xgb)
xgb_metrics["matthews_corrcoef"] = matthews_corrcoef(y_test, y_pred_xgb)
xgb_metrics["cohen_kappa"] = cohen_kappa_score(y_test, y_pred_xgb)

cm_xgb = confusion_matrix(y_test, y_pred_xgb)
xgb_metrics["confusion_matrix"] = cm_xgb.tolist()

report_xgb = classification_report(y_test, y_pred_xgb, output_dict=True, zero_division=0)
xgb_metrics["classification_report"] = report_xgb

fpr_xgb, tpr_xgb, _ = roc_curve(y_test, y_proba_xgb)
prec_curve_xgb, rec_curve_xgb, _ = precision_recall_curve(y_test, y_proba_xgb)
xgb_metrics["roc_curve"] = {"fpr": fpr_xgb.tolist(), "tpr": tpr_xgb.tolist()}
xgb_metrics["pr_curve"] = {"precision": prec_curve_xgb.tolist(), "recall": rec_curve_xgb.tolist()}

for k, v in xgb_metrics.items():
    if isinstance(v, float):
        print(f"{k:>22}: {v:.4f}")

print("\nConfusion Matrix:")
print(cm_xgb)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb, zero_division=0))

# ---------------------------------------------------------------------------
# 4. Feature importances
# ---------------------------------------------------------------------------
importances = xgb_model.feature_importances_
top_idx = np.argsort(importances)[::-1][:20]
print("\nTop 20 most important features:")
for i in top_idx:
    print(f"  {feature_names[i]:<40} {importances[i]:.4f}")

xgb_metrics["feature_importances"] = dict(zip(feature_names, importances.tolist()))

# ---------------------------------------------------------------------------
# 5. Save model + metrics
# ---------------------------------------------------------------------------
print(f"\nSaving model to {XGB_MODEL_OUT} ...")
with open(XGB_MODEL_OUT, "wb") as f:
    pickle.dump(
        {
            "model": xgb_model,
            "feature_names": feature_names,
        },
        f,
    )

print(f"Saving metrics to {XGB_METRICS_OUT} ...")
with open(XGB_METRICS_OUT, "wb") as f:
    pickle.dump(xgb_metrics, f)

print("Done.")

# ---------------------------------------------------------------------------
# 6. Comparison across all models trained so far, if their metrics dicts
#    are still in memory
# ---------------------------------------------------------------------------
try:
    print("\n" + "=" * 60)
    print("LR vs NAIVE BAYES vs SVM vs RANDOM FOREST vs XGBOOST")
    print("=" * 60)
    for k in ["accuracy", "precision", "recall", "f1_score", "roc_auc"]:
        row = f"{k:>12} | LR: {metrics[k]:.4f}"
        try:
            row += f" | NB: {nb_metrics[k]:.4f}"
        except NameError:
            pass
        try:
            row += f" | SVM: {svm_metrics[k]:.4f}"
        except NameError:
            pass
        try:
            row += f" | RF: {rf_metrics[k]:.4f}"
        except NameError:
            pass
        row += f" | XGB: {xgb_metrics[k]:.4f}"
        print(row)
except NameError:
    pass  # earlier metrics dicts aren't in memory — skip comparison

Training XGBoost...
Training took 190.7s

XGBOOST - EVALUATION METRICS
              accuracy: 0.9985
             precision: 0.9985
                recall: 0.9985
              f1_score: 0.9985
               roc_auc: 0.9999
     average_precision: 0.9999
              log_loss: 0.0085
     matthews_corrcoef: 0.9971
           cohen_kappa: 0.9971

Confusion Matrix:
[[4688    7]
 [   6 4236]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      4695
           1       1.00      1.00      1.00      4242

    accuracy                           1.00      8937
   macro avg       1.00      1.00      1.00      8937
weighted avg       1.00      1.00      1.00      8937


Top 20 most important features:
  content_tfidf__reuters                   0.3904
  content_tfidf__filessupport              0.0344
  content_tfidf__getty                     0.0151
  content_tfidf__internet                  0.0137
  content_tfidf__cen